# Lab 3: Physical Reality & The Feedback Loop

**Estimated Time:** 20 minutes

### 🎯 Learning Objectives:
* Experience the 'Scissors Gap': The immense time cost of moving from architecture to physical synthesis.
* Understand Chapter 7's Feedback Architecture: feeding physical limits back to the LLM.
* Package EDA physical evidence into the run archive.


## Step 1: Feeling the Scissors Gap (EDA Synthesis)
Architecture proxies verify latency in seconds. Physical synthesis takes minutes (Yosys) to weeks (OpenROAD). We will simulate this cost by streaming a mock Place & Route terminal execution.

🛠️ **YOUR TURN:** Run this cell and feel the visceral pain of waiting for physical verification.


In [ ]:
import tempfile, subprocess, time, json
from pathlib import Path

with open('.arch2_workshop_state.json', 'r') as f:
    state = json.load(f)
out_dir = Path(state['run_archive'])

print("Generating RTL for the surviving 16x16 candidate...")
eda_dir = Path(tempfile.mkdtemp(prefix="arch2_eda_"))
rtl_path = eda_dir / "mac.v"
rtl = """module mac (\n    input clk, input [7:0] a, input [7:0] b, input [15:0] p_sum, output reg [15:0] out\n);\n    always @(posedge clk) out <= p_sum + (a * b);\nendmodule"""
rtl_path.write_text(rtl)

print("Synthesizing standard cells in Yosys (Fast)...")
yosys_script = eda_dir / "synth.ys"
yosys_script.write_text(f"""read_verilog {rtl_path}\nhierarchy -check -top mac\nsynth -top mac\nabc -g gates\nstat""")
result = subprocess.run(["yosys", "-s", str(yosys_script)], capture_output=True, text=True)

print("\n⏳ EXECUTING PHYSICAL PLACE & ROUTE (OpenROAD simulation)...")
print("In reality, this takes 3 weeks. We simulate the Scissors Gap here:")
logs = [
    "[INFO] Reading DEF for floorplan...",
    "[INFO] Initializing global routing grids...",
    "[WARN] Congestion detected at MAC array center (34%).",
    "[INFO] Running detailed routing (Iteration 1)...",
    "[INFO] Running detailed routing (Iteration 4)...",
    "[ERROR] Setup violations found in clock tree! Retrying...",
    "[INFO] Legalizing standard cells...",
    "[INFO] Final DRC checks passed."
]
for log in logs:
    print(log)
    time.sleep(1.5)
print("✅ P&R Complete.\n")

mac_cells = 0
for line in result.stdout.splitlines():
    if "Number of cells:" in line:
        mac_cells = int(line.split()[-1])
        break

total_cells = mac_cells * (16 * 16)
print(f"Physical Standard Cell Area: {total_cells} logic gates")

# Save this directly to out_dir so it gets sealed in Lab 4!
eda_evidence = {
    "target_architecture": "balanced_16x16",
    "total_gate_count": total_cells,
    "physically_viable": total_cells <= 10000
}
(out_dir / "eda_synthesis.json").write_text(json.dumps(eda_evidence, indent=2))

if eda_evidence['physically_viable']:
    print("✅ PASS: Physically viable architecture. Evidence saved to run archive.")
else:
    print("❌ FAIL: Exceeded physical routing budget!")


🛑 **Instructor Checkpoint:** What happens when you get a `FAIL` after waiting 3 weeks for physical routing? This introduces Chapter 7: The Feedback Architecture.


## Step 2: The Feedback Architecture (Chapter 7)
If the architecture failed physical synthesis, we cannot just guess again. We encode the failure as an **EDA Failure Card** (a JSON vector) and feed it back to the LLM to learn from the proxy.


In [ ]:
import json

# This is what a Feedback Vector looks like.
feedback_payload = {
    "context": "Iteration 1 Physical Synthesis",
    "architecture": "balanced_16x16",
    "error_type": "AREA_VIOLATION",
    "measured_gates": 12000,
    "budget_gates": 10000,
    "instruction": "The generated array exceeded standard cell budget by 20%. Propose a new architecture that reduces MAC counts while maintaining the 90,000 cycle latency."
}
print("Feedback Payload to send to LLM:\n")
print(json.dumps(feedback_payload, indent=2))


## 📚 Further Reading
* **Chapter 7**: Feedback Architectures.
* **Chapter 8**: Running the Loop through different fidelity proxy layers.
